#  Satellite Coverage Analysis over Colombian Cities

##  Objective

Analyze how satellite coverage varies across major Colombian cities using real TLE data and orbital propagation.

We aim to answer:

- How many satellites pass over each city in a day?
- Is coverage uniform across Colombia?
- Which satellite groups dominate the traffic?

## Imports

In [ ]:
# 🛰️ Satellite Coverage Analysis over Colombian Cities

## 🎯 Objective

Analyze how satellite coverage varies across major Colombian cities using real TLE data and orbital propagation.

We aim to answer:

- How many satellites pass over each city in a day?
- Is coverage uniform across Colombia?
- Which satellite groups dominate the traffic?

## Load TLE

In [ ]:
def load_tle_file(filepath):
    sats = []

    with open(filepath, "r") as f:
        lines = [line.strip() for line in f if line.strip() != ""]

    i = 0
    while i < len(lines) - 2:
        name = lines[i]
        l1 = lines[i+1]
        l2 = lines[i+2]

        if l1.startswith("1") and l2.startswith("2"):
            sats.append({
                "name": name,
                "l1": l1,
                "l2": l2
            })
            i += 3
        else:
            i += 1

    return sats

sats = load_tle_file("../data/active.txt")
len(sats)

## Orbit propagation

In [ ]:
MU = 398600.4418

def mean_motion_to_semi_major_axis(n):
    n_rad = n * 2 * np.pi / 86400
    return (MU / n_rad**2)**(1/3)

def solve_kepler(M, e):
    E = M
    for _ in range(10):
        E = E - (E - e*np.sin(E) - M) / (1 - e*np.cos(E))
    return E

def propagate_satellite(sat, duration_minutes=720, step_seconds=120):

    parts = sat["l2"].split()

    i = np.radians(float(parts[2]))
    raan = np.radians(float(parts[3]))
    e = float("0." + parts[4])
    argp = np.radians(float(parts[5]))
    M0 = np.radians(float(parts[6]))
    n = float(parts[7])

    a = mean_motion_to_semi_major_axis(n)

    lats, lons = [], []

    for t in range(0, duration_minutes * 60, step_seconds):

        M = M0 + (n * 2*np.pi / 86400) * t
        E = solve_kepler(M, e)

        x = a * (np.cos(E) - e)
        y = a * np.sqrt(1 - e**2) * np.sin(E)

        X = x * np.cos(raan) - y * np.cos(i) * np.sin(raan)
        Y = x * np.sin(raan) + y * np.cos(i) * np.cos(raan)
        Z = y * np.sin(i)

        r = np.sqrt(X**2 + Y**2 + Z**2)
        lat = np.degrees(np.arcsin(Z / r))
        lon = np.degrees(np.arctan2(Y, X))

        lats.append(lat)
        lons.append(lon)

    return lats, lons

## Colombian cities

In [2]:
cities = {
    "Bogotá": (4.7110, -74.0721),
    "Medellín": (6.2442, -75.5812),
    "Cali": (3.4516, -76.5320),
    "Barranquilla": (10.9639, -74.7964),
    "Cartagena": (10.3910, -75.4794),
    "Bucaramanga": (7.1193, -73.1227),
    "Pereira": (4.8143, -75.6946),
    "Santa Marta": (11.2408, -74.1990),
    "Manizales": (5.0703, -75.5138),
    "Cúcuta": (7.8891, -72.4967)
}

## Detect satelletes over cities

In [ ]:
def satellites_over_city(sats, city_lat, city_lon, radius=3):

    passing = []

    for sat in sats[:800]:  # performance balance

        lats, lons = propagate_satellite(sat)

        for lat, lon in zip(lats, lons):
            if abs(lat - city_lat) < radius and abs(lon - city_lon) < radius:
                passing.append(sat["name"])
                break

    return passing

## Compute results

In [ ]:
results = {}

for city, (lat, lon) in cities.items():
    passing = satellites_over_city(sats, lat, lon)
    results[city] = len(passing)

results

## Visualization

In [ ]:
df = pd.DataFrame(list(results.items()), columns=["City", "Satellites"])

df = df.sort_values(by="Satellites", ascending=False)

plt.figure(figsize=(10,5))
plt.bar(df["City"], df["Satellites"])
plt.xticks(rotation=45)
plt.ylabel("Number of Satellites")
plt.title("Satellite Coverage over Colombian Cities")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

## Insights

## 📊 Key Insights

- Satellite coverage is relatively uniform across major Colombian cities.
- Differences are minor due to the global nature of LEO constellations.
- Most observed satellites belong to large constellations such as Starlink.
- Cities closer to the equator experience frequent satellite passes due to orbital inclination patterns.

## 🚀 Interpretation

The results suggest that satellite internet and observation coverage over Colombia is dense and consistent, making it suitable for nationwide applications such as connectivity and Earth observation.